# Polylines and streamlines

**Geometry types:** `polyline` · `streamline`

Ordered vertex sequences. `streamline` marks tractography data; the read and
write functions are the same for both.

1. Generate synthetic streamlines
2. Write them with groups and per-object and per-vertex attributes
3. Open the store and inspect its metadata
4. Read all, by object id, by group, and by object attribute
5. Bounding-box query
6. Export to TRK
7. Validate

Building a streamline pyramid is in `11_tractography.ipynb`. The streamline
pyramid needs `fragment_attributes/segment_id`, which the converters
(`ingest_trk`, `ingest_tck`, `ingest_trx`) write and `write_polylines` does
not, so build pyramids on converted stores.

In [1]:
import os
import tempfile

import numpy as np
import zarr_vectors as zv
from zarr_vectors.types.polylines import read_polylines, write_polylines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE = os.path.join(_tmpdir, "tracts.zv")
print("Store path:", STORE)

Store path: /tmp/zv_examples_odyqdhdx/tracts.zv


## 1 · Generate synthetic streamlines

2,000 smooth random paths with 1 mm steps, starting inside a 120 mm cube
centred on the origin. Coordinates are in mm.

In [2]:
rng = np.random.default_rng(1)
N = 2_000

def random_streamline(n_steps, rng):
    direction = rng.normal(size=3)
    steps = []
    for _ in range(n_steps):
        direction += rng.normal(0, 0.15, 3)        # small turn per step
        direction /= np.linalg.norm(direction)
        steps.append(direction.copy())             # 1 mm step
    start = rng.uniform(-60, 60, 3)
    return (start + np.vstack([np.zeros(3), np.cumsum(steps, axis=0)])).astype(np.float32)

streamlines = [random_streamline(int(rng.integers(30, 100)), rng) for _ in range(N)]

# One value per streamline (object attributes) ...
lengths = np.array([np.linalg.norm(np.diff(s, axis=0), axis=1).sum() for s in streamlines],
                   dtype=np.float32)
mean_fa = rng.uniform(0.2, 0.7, N).astype(np.float32)
# ... and one value per vertex (a list with one array per streamline).
per_vertex_fa = [rng.uniform(0.1, 0.9, len(s)).astype(np.float32) for s in streamlines]

# Three bundles: group id -> object ids.
bundle_size = N // 3
groups = {
    0: list(range(bundle_size)),
    1: list(range(bundle_size, 2 * bundle_size)),
    2: list(range(2 * bundle_size, N)),
}
group_names = ["CST", "AF", "UF"]

print(f"{N:,} streamlines, {sum(len(s) for s in streamlines):,} vertices")
print(f"length range: {lengths.min():.1f} – {lengths.max():.1f} mm")
print(f"groups: {[(group_names[g], len(ids)) for g, ids in groups.items()]}")

2,000 streamlines, 130,631 vertices
length range: 30.0 – 99.0 mm
groups: [('CST', 666), ('AF', 666), ('UF', 668)]


## 2 · Write with groups and attributes

A streamline that crosses chunk boundaries is cut into one fragment per chunk
it passes through, joined by cross-chunk links.

In [3]:
summary = write_polylines(
    STORE,
    streamlines,
    chunk_shape=(25.0, 25.0, 25.0),   # mm
    bin_shape=(5.0, 5.0, 5.0),
    geometry_type="streamline",
    groups=groups,
    object_attributes={"length": lengths, "mean_fa": mean_fa},
    vertex_attributes={"fa": per_vertex_fa},
)
print(summary)

{'polyline_count': 2000, 'vertex_count': 130631, 'chunk_count': 608, 'cross_chunk_link_count': 7705, 'group_count': 3}


## 3 · Open and inspect

In [4]:
ds = zv.open(STORE)
lv0 = ds.level(0)
print(ds)
print(f"chunk shape  : {lv0.scale} mm")
print(f"objects      : {lv0.objects.count:,}")
print(f"vertex_count : {lv0.vertex_count:,}")
print(f"groups       : {lv0.groups.names()}")
print(f"vertex attrs : {lv0.attribute_names('vertex')}")
print(f"object attrs : {lv0.attribute_names('object')}")
print(f"bounds       : {ds.bounds[0].round(1)} → {ds.bounds[1].round(1)} mm")

Dataset('file:///tmp/zv_examples_odyqdhdx/tracts.zv', streamline, levels=[0])
chunk shape  : (25.0, 25.0, 25.0) mm
objects      : 2,000
vertex_count : 130,631
groups       : ('group_0', 'group_1', 'group_2')
vertex attrs : ('fa',)
object attrs : ('length', 'mean_fa')
bounds       : [-131.7 -125.9 -130.9] → [129.6 125.2 130.4] mm


## 4a · Read all streamlines

`polylines[i]` is a list of arrays, one per chunk run of streamline *i*;
concatenate them for the whole path.

In [5]:
result_all = read_polylines(STORE)
print(f"polyline_count : {result_all['polyline_count']:,}")
print(f"vertex_count   : {result_all['vertex_count']:,}")

for i in (0, 1):
    runs = result_all["polylines"][i]
    path = np.concatenate(runs)
    print(f"streamline {i}: {len(path)} vertices in {len(runs)} chunk runs, "
          f"matches input: {np.allclose(path, streamlines[i])}")

polyline_count : 2,000
vertex_count   : 130,631
streamline 0: 64 vertices in 6 chunk runs, matches input: True
streamline 1: 66 vertices in 4 chunk runs, matches input: True


## 4b · Read by object id

In [6]:
result_id = read_polylines(STORE, object_ids=[0, 42, 999])
print(f"read {result_id['polyline_count']} streamlines, ids {result_id['object_ids']}")
sl42 = np.concatenate(result_id["polylines"][1])
print(f"streamline 42: {len(sl42)} vertices (input: {len(streamlines[42])})")

read 3 streamlines, ids [0, 42, 999]
streamline 42: 90 vertices (input: 90)


## 4c · Read by group

`groups` keys are integers; the store names them `group_0`, `group_1`, ….

In [7]:
for gid, name in enumerate(group_names):
    r = read_polylines(STORE, group_ids=[gid])
    print(f"group {gid} ({name}): {r['polyline_count']:,} streamlines")

group 0 (CST): 666 streamlines


group 1 (AF): 666 streamlines


group 2 (UF): 668 streamlines


## 4d · Select by object attribute

Object attributes are small, one value per streamline, so read them without
the geometry, pick ids, then read only those streamlines.

In [8]:
from zarr_vectors.building import get_resolution_level, open_store, read_object_attributes

level0 = get_resolution_level(open_store(STORE), 0)
lengths_stored = read_object_attributes(level0, "length")
fa_stored = read_object_attributes(level0, "mean_fa")
print(f"mean_fa range: [{fa_stored.min():.3f}, {fa_stored.max():.3f}]")
print(f"length range : [{lengths_stored.min():.1f}, {lengths_stored.max():.1f}] mm")

selected = np.flatnonzero((fa_stored > 0.55) & (lengths_stored > 60))
print(f"\nmean_fa > 0.55 and length > 60 mm: {len(selected)} streamlines")
result_sel = read_polylines(STORE, object_ids=selected[:10].tolist())
print(f"read the first {result_sel['polyline_count']} of them")

mean_fa range: [0.201, 0.700]
length range : [30.0, 99.0] mm

mean_fa > 0.55 and length > 60 mm: 347 streamlines
read the first 10 of them


## 5 · Bounding-box query

`bbox=` selects by chunk: it returns, whole, every streamline that passes
through a chunk the box touches. The 20 mm box below touches the eight 25 mm
chunks around the origin, a 50 mm cube. For the streamlines that enter the box
itself, test the vertices of the result.

In [9]:
lo = np.array([-10.0, -10.0, -10.0])
hi = np.array([ 10.0,  10.0,  10.0])

bbox_result = read_polylines(STORE, bbox=(lo, hi))
paths = [np.concatenate(runs) for runs in bbox_result["polylines"]]
enters = [((p >= lo) & (p <= hi)).all(axis=1).any() for p in paths]

print(f"streamlines through the touched chunks : {bbox_result['polyline_count']:,}")
print(f"  of which enter the 20 mm box          : {sum(enters):,}")

truth = sum(bool(((s >= lo) & (s <= hi)).all(axis=1).any()) for s in streamlines)
print(f"input streamlines that enter the box    : {truth:,}")

streamlines through the touched chunks : 373
  of which enter the 20 mm box          : 60
input streamlines that enter the box    : 60


## 6 · Export to TRK

`export_trk` writes positions, per-vertex attributes as TRK scalars and the
object attributes named in `object_attribute_names` as TRK properties.

In [10]:
import nibabel as nib
from zarr_vectors_tools.convert.export.trk import export_trk

trk_path = os.path.join(_tmpdir, "tracts.trk")
export_trk(STORE, trk_path, attribute_names=["fa"], object_attribute_names=["length"])

trk = nib.streamlines.load(trk_path)
print(f"TRK streamlines : {len(trk.streamlines):,}")
print(f"scalars         : {list(trk.tractogram.data_per_point)}")
print(f"properties      : {list(trk.tractogram.data_per_streamline)}")
print(f"first streamline matches input: {np.allclose(trk.streamlines[0], streamlines[0], atol=1e-4)}")

TRK streamlines : 2,000
scalars         : ['fa']
properties      : ['length']
first streamline matches input: True


## 7 · Validate

In [11]:
print(validate(STORE, level=3).summary())

Level 3 validation: PASS
  27 passed, 0 warnings, 0 errors


## Summary

| Step | API |
|------|-----|
| Write | `write_polylines(path, polylines, chunk_shape, bin_shape, geometry_type="streamline", groups=..., object_attributes=..., vertex_attributes=...)` |
| Read all | `read_polylines(path)` → `polylines[i]` is a list of chunk runs |
| Read by id | `read_polylines(path, object_ids=[...])` |
| Read by group | `read_polylines(path, group_ids=[...])` |
| Bbox query | `read_polylines(path, bbox=(lo, hi))` (whole streamlines through the touched chunks) |
| Object attributes | `zarr_vectors.building.read_object_attributes(level_group, name)` |
| Export | `export_trk(path, out.trk, attribute_names=..., object_attribute_names=...)` |